# Pandas from First Principles
## Notebook 2: The DataFrame

---

**What you already know (Notebook 1):** Series, Index, `.loc`/`.iloc`, vectorized operations, missing data.

**What you will learn here:** The DataFrame — how to create it, inspect it, select from it, and modify it.

**Topics strictly covered in this notebook:**
- What a DataFrame is
- Creating a DataFrame
- Inspecting a DataFrame
- Selecting columns
- Selecting rows
- Adding new columns
- Removing columns
- Renaming columns
- Basic filtering

---

### Table of Contents

1. What is a DataFrame?
2. Creating a DataFrame
3. Inspecting a DataFrame
4. Selecting Columns
5. Selecting Rows
6. Filtering Rows
7. Adding New Columns
8. Removing Columns
9. Renaming Columns
10. Practice Questions

---

# 1. What is a DataFrame?

## What is it?

A DataFrame is a **two-dimensional table** with rows and columns — exactly like a spreadsheet or a database table.

Every column in a DataFrame is a **Series**. They all share the same Index (row labels).

```
         name    age    city
0       Alice     25   Delhi      ← row 0
1         Bob     30  Mumbai      ← row 1
2       Carol     27    Pune      ← row 2

↑ index   ↑ column 1  ↑ column 2  ↑ column 3
```

## Why do we need it?

A Series holds one column of data. But real data has many columns — name, age, salary, city, department — all related to the same set of people or records.

A DataFrame keeps all these columns together under one roof, with a shared index.

## Real-world analogy

Think of a DataFrame as an Excel sheet:
- **Rows** = individual records (one person, one order, one transaction)
- **Columns** = attributes (name, age, salary)
- **Index** = the row number (or label) on the left side

## Internal structure

A DataFrame is internally stored as a **dictionary of Series objects**:

```
DataFrame
  ├── 'name'  → Series(['Alice', 'Bob', 'Carol'])
  ├── 'age'   → Series([25, 30, 27])
  └── 'city'  → Series(['Delhi', 'Mumbai', 'Pune'])
```

All Series share the same index. This is why each column can have a **different dtype**, but within a column, all values must be the same type.

---

# 2. Creating a DataFrame

## Syntax

```python
pd.DataFrame(data, index=None, columns=None)
```

## Parameters

| Parameter | What it does |
|---|---|
| `data` | The input data — dict, list of dicts, list of lists, NumPy array, or another DataFrame |
| `index` | Row labels. Auto-assigned (0, 1, 2...) if not given |
| `columns` | Column names. Useful when `data` is a list of lists |

## Return value

Returns a `pandas.DataFrame` object. Original data is not mutated.

---

## 2.1 From a dictionary of lists

This is the most common way to create a DataFrame manually.

**Rule:** Each key becomes a **column name**. Each list becomes the **column values**. All lists must be the same length.

In [ ]:
import pandas as pd

# Each key = column name, each list = column values
employee_data = {
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "age":        [28, 34, 26, 41, 30],
    "department": ["Engineering", "Marketing", "Engineering", "HR", "Marketing"],
    "salary":     [85000, 72000, 91000, 68000, 75000],
}

employees = pd.DataFrame(employee_data)
print(employees)

Notice:
- Index is auto-assigned as 0, 1, 2, 3, 4
- Columns appear in the same order as the dictionary keys
- Each column has its own dtype (name/department = object, age/salary = int64)

## 2.2 From a list of dictionaries

Each dictionary in the list becomes **one row**. Keys become column names.

This is what you get when you load JSON data or an API response.

In [ ]:
# Each dict = one row
orders = [
    {"order_id": 101, "product": "Laptop",  "quantity": 1, "price": 75000},
    {"order_id": 102, "product": "Mouse",   "quantity": 3, "price": 1200},
    {"order_id": 103, "product": "Keyboard","quantity": 2, "price": 2500},
    {"order_id": 104, "product": "Monitor", "quantity": 1, "price": 22000},
]

orders_df = pd.DataFrame(orders)
print(orders_df)

## 2.3 What happens when dictionaries have missing keys?

Just like the Series alignment behavior — missing keys become `NaN`.

In [ ]:
# Row 3 is missing 'discount' key
# Row 1 is missing 'notes' key
messy_data = [
    {"product": "Laptop",   "price": 75000, "discount": 5000},
    {"product": "Mouse",    "price": 1200,  "discount": 100,  "notes": "bestseller"},
    {"product": "Monitor",  "price": 22000},
]

messy_df = pd.DataFrame(messy_data)
print(messy_df)
print("\nMissing values appear as NaN automatically")

## 2.4 From a list of lists

When you have raw rows without column names, use `columns` parameter.

In [ ]:
# Raw rows — you must provide column names separately
raw_rows = [
    ["Alice", 28, "Engineering", 85000],
    ["Bob",   34, "Marketing",   72000],
    ["Carol", 26, "Engineering", 91000],
]

col_names = ["name", "age", "department", "salary"]

df_from_rows = pd.DataFrame(raw_rows, columns=col_names)
print(df_from_rows)

## 2.5 Setting a custom index

By default, the index is 0, 1, 2... But you can use any column as the index.

In [ ]:
# Method 1: Pass index when creating
students = pd.DataFrame(
    {
        "name":  ["Riya", "Arjun", "Priya"],
        "score": [88, 74, 91],
        "grade": ["A", "B", "A"],
    },
    index=["S001", "S002", "S003"]   # custom row labels
)
print("Custom index:")
print(students)

In [ ]:
# Method 2: Set an existing column as index using set_index()
employees = pd.DataFrame({
    "emp_id":     ["E01", "E02", "E03", "E04"],
    "name":       ["Alice", "Bob", "Carol", "Dave"],
    "salary":     [85000, 72000, 91000, 68000],
})

# Make emp_id the index — now rows are labeled by employee ID
employees_indexed = employees.set_index("emp_id")
print(employees_indexed)

**Note:** `set_index()` returns a **new** DataFrame. The original is unchanged. If you want to reverse this, use `.reset_index()` — it converts the index back into a column.

---

# 3. Inspecting a DataFrame

## What is it?

Before doing any analysis, you need to **understand your data**. These methods give you a quick overview of what you are working with.

This is always the **first thing** you do when you load a new dataset.

In [ ]:
import pandas as pd

# We will use this DataFrame for all examples in this section
employees = pd.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
    "age":        [28, 34, 26, 41, 30, 35],
    "department": ["Engineering", "Marketing", "Engineering", "HR", "Marketing", "Engineering"],
    "salary":     [85000, 72000, 91000, 68000, 75000, 95000],
    "city":       ["Delhi", "Mumbai", "Pune", "Delhi", "Bangalore", "Mumbai"],
})

print(employees)

## 3.1 `.head()` and `.tail()` — see a few rows

In [ ]:
# .head(n) — first n rows. Default is 5
print("First 3 rows:")
print(employees.head(3))

In [ ]:
# .tail(n) — last n rows. Default is 5
print("Last 2 rows:")
print(employees.tail(2))

**When to use:** Always use `.head()` right after loading a dataset. It shows you the structure without printing thousands of rows.

## 3.2 `.shape` — dimensions of the DataFrame

In [ ]:
# .shape returns (rows, columns) — a tuple
print("Shape:", employees.shape)

num_rows, num_cols = employees.shape
print(f"Rows: {num_rows}, Columns: {num_cols}")

## 3.3 `.dtypes` — data type of each column

In [ ]:
# .dtypes shows you the dtype of every column
print(employees.dtypes)

**Why this matters:** The dtype tells you what operations are valid. If `salary` shows as `object` instead of `int64`, you cannot do arithmetic on it. You would need to fix it with `astype()` or `pd.to_numeric()`.

**Common situation:** After loading a CSV, many columns show as `object` because the file had mixed or messy data. Always check `.dtypes` first.

## 3.4 `.info()` — complete overview in one call

In [ ]:
# .info() gives you shape, dtypes, AND missing value count — all at once
employees.info()

Read `.info()` output like this:

```
RangeIndex: 6 entries     → 6 rows
Data columns: 5 total     → 5 columns

name        6 non-null    → 0 missing values (all 6 rows have data)
salary      6 non-null    → 0 missing values
```

If you had missing values, the count would be less than total rows. For example, `4 non-null` in a 6-row DataFrame means 2 values are missing.

**Interview note:** `.info()` is the single most useful method when you first receive a dataset. It answers: how many rows? how many columns? what are the types? where is data missing? — all in one shot.

## 3.5 `.describe()` — statistics for numeric columns

In [ ]:
# .describe() gives count, mean, std, min, max etc. for numeric columns only
print(employees.describe())

In [ ]:
# To include non-numeric (string) columns, pass include='all'
print(employees.describe(include="all"))

## 3.6 `.columns` and `.index` — see labels

In [ ]:
# Column names
print("Columns:", employees.columns.tolist())

# Row index
print("Index:  ", employees.index.tolist())

---

# 4. Selecting Columns

## What is it?

Selecting a column means pulling out one or more columns from a DataFrame. The result is either a **Series** (one column) or a **DataFrame** (multiple columns).

## Syntax

```python
df["column_name"]          # single column → returns Series
df[["col1", "col2"]]       # multiple columns → returns DataFrame
```

## 4.1 Selecting a single column

In [ ]:
# Single column — returns a Series
salaries = employees["salary"]

print(salaries)
print("\nType:", type(salaries))   # pandas.Series

**Important:** `df["salary"]` returns a **Series**, not a DataFrame. The Series shares the same index as the DataFrame.

Once you have a Series, all the Series methods from Notebook 1 apply — `.mean()`, `.max()`, `.isna()`, `.value_counts()`, etc.

In [ ]:
# Once you have a Series, use Series methods on it
print("Average salary:", employees["salary"].mean())
print("Highest salary:", employees["salary"].max())
print("Departments:   ", employees["department"].value_counts().to_dict())

## 4.2 Selecting multiple columns

In [ ]:
# Multiple columns — use a list of names — returns a DataFrame
name_and_salary = employees[["name", "salary"]]

print(name_and_salary)
print("\nType:", type(name_and_salary))   # pandas.DataFrame

**Common mistake — the double bracket confusion:**

```python
df["salary"]          # Single bracket → Series
df[["salary"]]        # Double bracket → DataFrame with 1 column
df[["name","salary"]] # Double bracket → DataFrame with 2 columns
```

The outer `[]` is the selection operator. The inner `[]` is a Python list of column names.

In [ ]:
# Demonstrating the difference
single = employees["salary"]
double = employees[["salary"]]

print("Single bracket type:", type(single))   # Series
print("Double bracket type:", type(double))   # DataFrame
print()
print("Single bracket (Series):")
print(single.values)  # just the values
print("\nDouble bracket (DataFrame):")
print(double)

## 4.3 Dot notation — and why to avoid it

In [ ]:
# You can also access a column like an attribute (dot notation)
print(employees.salary.mean())

# This works but has problems:
# 1. Fails if column name has spaces: df.first name → SyntaxError
# 2. Conflicts with DataFrame methods: df.count refers to the method, not a column named 'count'
# 3. Cannot be used for assignment: df.new_col = ... is unreliable

print("\nPrefer df['salary'] over df.salary — it always works.")

**Best practice:** Always use `df["column_name"]`. Never rely on dot notation for production code.

---

# 5. Selecting Rows

## What is it?

Selecting rows means pulling out specific rows from a DataFrame. Just like with Series, we use `.loc` for label-based access and `.iloc` for position-based access.

## Syntax

```python
df.loc[row_label]              # single row by label
df.loc[row_label, col_name]    # specific row AND column
df.iloc[row_position]          # single row by position
df.iloc[row_pos, col_pos]      # specific row AND column by position
```

## 5.1 `.loc[]` — row by label

In [ ]:
print("Full DataFrame (for reference):")
print(employees)
print()

In [ ]:
# Single row by label — returns a Series (the row becomes a Series)
print("Row with label 0:")
print(employees.loc[0])
print("\nType:", type(employees.loc[0]))

In [ ]:
# Multiple rows by label — returns a DataFrame
print("Rows 1 and 3:")
print(employees.loc[[1, 3]])

In [ ]:
# Slice of rows by label — INCLUSIVE on both ends
print("Rows 1 to 3 (inclusive):")
print(employees.loc[1:3])

## 5.2 Selecting specific rows AND columns with `.loc`

`.loc` takes two arguments: `df.loc[rows, columns]`

In [ ]:
# Row 2, column 'name'
print("Row 2, name column:")
print(employees.loc[2, "name"])

# Rows 0-2, columns 'name' and 'salary'
print("\nRows 0-2, name and salary:")
print(employees.loc[0:2, ["name", "salary"]])

## 5.3 `.iloc[]` — row by position

In [ ]:
# First row — position 0
print("First row (position 0):")
print(employees.iloc[0])

In [ ]:
# Last row
print("Last row:")
print(employees.iloc[-1])

# First 3 rows — EXCLUSIVE on right end (like Python slicing)
print("\nFirst 3 rows:")
print(employees.iloc[0:3])

In [ ]:
# Row position 1, column position 3 (salary column)
print("Row 1, column 3 (salary):", employees.iloc[1, 3])

# First 3 rows, first 2 columns
print("\nFirst 3 rows, first 2 columns:")
print(employees.iloc[0:3, 0:2])

## 5.4 Summary — `.loc` vs `.iloc` for DataFrames

| | `.loc` | `.iloc` |
|---|---|---|
| **Rows** | By index label | By integer position |
| **Columns** | By column name | By integer position |
| **Slice end** | Inclusive | Exclusive |
| **Use when** | You know the label | You know the position |

The behavior is exactly the same as in Series (Notebook 1) — it just now works in two dimensions.

---

# 6. Filtering Rows

## What is it?

Filtering means selecting only the rows that meet a certain condition. This is the most common operation in data analysis.

The idea is exactly the same as boolean indexing in Series (Notebook 1) — just applied to a DataFrame.

## How it works

Step 1: Create a boolean condition on a column — produces a True/False Series.
Step 2: Use that mask to filter the DataFrame — only True rows are returned.

In [ ]:
# Step by step — so you can see exactly what happens

# Step 1: Create the condition
high_salary_mask = employees["salary"] > 80000
print("Boolean mask:")
print(high_salary_mask)
print()

In [ ]:
# Step 2: Apply the mask to the DataFrame
high_earners = employees[high_salary_mask]
print("Employees with salary > 80,000:")
print(high_earners)

In [ ]:
# In practice: write it in one line
print(employees[employees["salary"] > 80000])

## 6.1 Filtering on string columns

In [ ]:
# Filter by exact string match
engineers = employees[employees["department"] == "Engineering"]
print("Engineering department:")
print(engineers)

## 6.2 Combining multiple conditions

In [ ]:
# AND condition: Engineering department AND salary > 85000
# Remember: use & not 'and', and wrap each condition in ()

senior_engineers = employees[
    (employees["department"] == "Engineering") & (employees["salary"] > 85000)
]
print("Senior Engineers (dept=Engineering AND salary > 85000):")
print(senior_engineers)

In [ ]:
# OR condition: Marketing OR HR department
non_tech = employees[
    (employees["department"] == "Marketing") | (employees["department"] == "HR")
]
print("Marketing or HR:")
print(non_tech)

In [ ]:
# Better way for OR on same column: use isin()
non_tech_clean = employees[employees["department"].isin(["Marketing", "HR"])]
print("Same result using isin():")
print(non_tech_clean)

## 6.3 NOT condition

In [ ]:
# ~ means NOT — invert the condition
not_engineering = employees[~(employees["department"] == "Engineering")]
print("Everyone except Engineering:")
print(not_engineering)

---

# 7. Adding New Columns

## What is it?

You can add a new column to a DataFrame by assigning to `df["new_column_name"]`.

The new column is usually computed from existing columns.

## Important note

This **modifies the DataFrame in place** — unlike most other Pandas operations. The original DataFrame changes.

In [ ]:
# Adding a column by assigning to a new key
# This computes annual salary as a new column

employees["annual_bonus"] = employees["salary"] * 0.10

print(employees)

In [ ]:
# Adding a column derived from string operations
employees["name_upper"] = employees["name"].str.upper()

print(employees[["name", "name_upper"]])

In [ ]:
# Adding a boolean column — is the employee senior (age > 30)?
employees["is_senior"] = employees["age"] > 30

print(employees[["name", "age", "is_senior"]])

In [ ]:
# Adding a constant column — same value for all rows
employees["company"] = "TechCorp"

print(employees[["name", "company"]].head())

## Using `.assign()` — the cleaner alternative

`.assign()` adds a column and returns a **new** DataFrame (non-mutating). It is the preferred way when you are method chaining.

In [ ]:
# .assign() — returns a new DataFrame, does not modify original
employees_with_tax = employees.assign(
    salary_after_tax=employees["salary"] * 0.70  # 30% tax
)

print(employees_with_tax[["name", "salary", "salary_after_tax"]])

---

# 8. Removing Columns

## What is it?

You can remove columns from a DataFrame using `.drop()`.

## Syntax

```python
df.drop(labels, axis=1)       # drop columns (axis=1 means columns)
df.drop(labels, axis=0)       # drop rows (axis=0 means rows)
```

## Parameters

| Parameter | Purpose |
|---|---|
| `labels` | Column name(s) or row label(s) to drop |
| `axis` | `1` for columns, `0` for rows |
| `inplace` | `False` (default) = return new DataFrame. `True` = modify in place |

## Return value

Returns a **new** DataFrame by default. Original is unchanged unless `inplace=True`.

In [ ]:
# Let's start fresh to keep it clean
employees_clean = pd.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "age":        [28, 34, 26, 41, 30],
    "department": ["Engineering", "Marketing", "Engineering", "HR", "Marketing"],
    "salary":     [85000, 72000, 91000, 68000, 75000],
    "city":       ["Delhi", "Mumbai", "Pune", "Delhi", "Bangalore"],
})

# Drop a single column — returns a new DataFrame
without_city = employees_clean.drop("city", axis=1)
print("After dropping 'city':")
print(without_city)

In [ ]:
# Drop multiple columns at once
without_age_city = employees_clean.drop(["age", "city"], axis=1)
print("After dropping 'age' and 'city':")
print(without_age_city)

In [ ]:
# Confirm original is unchanged (drop is non-mutating by default)
print("Original still has all columns:", employees_clean.columns.tolist())

In [ ]:
# Drop a row by index label
without_row_2 = employees_clean.drop(2, axis=0)   # axis=0 means rows
print("After dropping row 2:")
print(without_row_2)

**Best practice on `inplace`:**

Avoid `inplace=True`. It makes code harder to read and debug. Instead, reassign:

```python
# Avoid this:
df.drop("city", axis=1, inplace=True)

# Prefer this:
df = df.drop("city", axis=1)
```

The second form is clearer — you can see that `df` is being reassigned.

---

# 9. Renaming Columns

## What is it?

Renaming columns means changing their names — useful when you load data with messy, abbreviated, or unclear column names.

## Syntax

```python
df.rename(columns={"old_name": "new_name"})
```

## Return value

Returns a **new** DataFrame. Original unchanged.

In [ ]:
# Simulating a DataFrame with messy column names from a CSV
raw_data = pd.DataFrame({
    "emp_nm":  ["Alice", "Bob", "Carol"],
    "emp_age": [28, 34, 26],
    "dept":    ["Engineering", "Marketing", "Engineering"],
    "sal":     [85000, 72000, 91000],
})
print("Before rename:")
print(raw_data)

In [ ]:
# Rename specific columns using a dict
# {"old_name": "new_name"}
clean_data = raw_data.rename(columns={
    "emp_nm":  "name",
    "emp_age": "age",
    "dept":    "department",
    "sal":     "salary",
})

print("After rename:")
print(clean_data)

In [ ]:
# Renaming ALL columns at once — replace the full list
df = pd.DataFrame([[1, 2, 3], [4, 5, 6]], columns=["a", "b", "c"])
df.columns = ["col_one", "col_two", "col_three"]   # assign new list directly
print(df)

In [ ]:
# Practical: remove spaces and lowercase all column names in one step
# This is a very common operation when loading CSV files

messy_columns = pd.DataFrame(columns=["First Name", "Last Name", "Emp Age", "Annual Salary"])

# Convert all to lowercase and replace spaces with underscores
messy_columns.columns = messy_columns.columns.str.lower().str.replace(" ", "_")

print("Cleaned column names:", messy_columns.columns.tolist())

**Best practice:** Always clean column names early in your pipeline. Lowercase with underscores is the standard Python convention. It prevents errors and makes code easier to type.

---

# 10. Practice Questions

All questions are **Easy to Medium** level.
Each question covers exactly what was taught in this notebook.

---

## Level 1 — Easy

### Q1 — Create and Inspect

Create a DataFrame called `books` with the following data:

| title | author | year | price | in_stock |
|---|---|---|---|---|
| Python Crash Course | Eric Matthes | 2019 | 499 | True |
| Clean Code | Robert Martin | 2008 | 699 | False |
| The Pragmatic Programmer | Hunt & Thomas | 1999 | 799 | True |
| Fluent Python | Luciano Ramalho | 2022 | 899 | True |
| Head First Python | Paul Barry | 2016 | 599 | False |

Then:
- a) Print the shape
- b) Print the dtypes
- c) Print first 3 rows using `.head()`
- d) Use `.info()` to check for missing values

In [ ]:
import pandas as pd

# Q1 — Your solution here


### Q2 — Column Selection

Using the `books` DataFrame from Q1:
- a) Select only the `title` column — what type is returned?
- b) Select `title` and `price` columns together — what type is returned?
- c) Find the **average price** of all books
- d) Find the **most expensive** book's price

In [ ]:
# Q2 — Your solution here


### Q3 — Row Selection

Using the `books` DataFrame:
- a) Select the **second row** using `.iloc`
- b) Select the **last row** using `.iloc`
- c) Select rows **2 to 4** using `.loc`
- d) Select the `title` of row 0 using `.loc[row, column]`

In [ ]:
# Q3 — Your solution here


### Q4 — Basic Filtering

Using the `books` DataFrame:
- a) Filter books that are **in stock**
- b) Filter books with **price below 700**
- c) Filter books published **after 2015**

In [ ]:
# Q4 — Your solution here


---

## Level 2 — Medium

### Q5 — Adding and Removing Columns

Using the `books` DataFrame:
- a) Add a new column `discounted_price` = price with 15% discount
- b) Add a boolean column `is_recent` = books published after 2015
- c) Drop the `in_stock` column
- d) Print the final DataFrame

In [ ]:
# Q5 — Your solution here


### Q6 — Rename and Clean Columns

This DataFrame has messy column names — fix them.

- a) Rename `Prod Name` → `product_name`, `Unit Price` → `unit_price`, `Qty Sold` → `quantity_sold`
- b) Add a new column `total_revenue` = `unit_price` × `quantity_sold`
- c) Filter only rows where `total_revenue` > 10000
- d) Sort the result by `total_revenue` from high to low

In [ ]:
sales_raw = pd.DataFrame({
    "Prod Name":  ["Laptop", "Mouse", "Monitor", "Keyboard", "Webcam"],
    "Unit Price": [75000, 1200, 22000, 2500, 4500],
    "Qty Sold":   [5, 120, 8, 45, 30],
})

# Q6 — Your solution here


### Q7 — Multi-condition Filtering

Using the employee data below:
- a) Find all employees in **Engineering** with salary **above 80,000**
- b) Find employees from **Delhi or Mumbai**
- c) Find employees who are **NOT in Marketing**
- d) Find employees aged between **25 and 35** (inclusive)

In [ ]:
team = pd.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "age":        [28, 34, 26, 41, 30, 35, 27],
    "department": ["Engineering", "Marketing", "Engineering", "HR", "Marketing", "Engineering", "HR"],
    "salary":     [85000, 72000, 91000, 68000, 75000, 95000, 64000],
    "city":       ["Delhi", "Mumbai", "Pune", "Delhi", "Bangalore", "Mumbai", "Pune"],
})

# Q7a — Engineering AND salary > 80,000

# Q7b — Delhi or Mumbai

# Q7c — NOT Marketing

# Q7d — Age between 25 and 35


### Q8 — Build a complete mini pipeline

You receive the following student data. Do all steps in order:

- a) Rename columns: `nm` → `name`, `mks` → `marks`, `cls` → `class_section`
- b) Add a column `grade`: `'A'` if marks >= 85, `'B'` if 70–84, `'C'` if below 70
  - Hint: use `.apply()` with a custom function
- c) Filter only students who got grade `'A'`
- d) Drop the `marks` column from the filtered result
- e) Print the final output

In [ ]:
students_raw = pd.DataFrame({
    "nm":  ["Riya", "Arjun", "Priya", "Karan", "Meera", "Dev", "Sana"],
    "mks": [91, 74, 88, 62, 85, 78, 95],
    "cls": ["10A", "10B", "10A", "10C", "10B", "10A", "10C"],
})

# Q8 — Your solution here (do each step one at a time)


---

## Answer Key

In [ ]:
# ── Q1 Answer ────────────────────────────────────────────────────────────────
books = pd.DataFrame({
    "title":    ["Python Crash Course", "Clean Code", "The Pragmatic Programmer",
                 "Fluent Python", "Head First Python"],
    "author":   ["Eric Matthes", "Robert Martin", "Hunt & Thomas",
                 "Luciano Ramalho", "Paul Barry"],
    "year":     [2019, 2008, 1999, 2022, 2016],
    "price":    [499, 699, 799, 899, 599],
    "in_stock": [True, False, True, True, False],
})

print("a) Shape:", books.shape)
print("b) dtypes:\n", books.dtypes)
print("c) First 3 rows:\n", books.head(3))
print("d) Info:")
books.info()

In [ ]:
# ── Q2 Answer ────────────────────────────────────────────────────────────────
print("a) Single column type:", type(books["title"]))          # Series
print("b) Multi column type: ", type(books[["title","price"]])  # DataFrame
print("c) Average price:     ", books["price"].mean())
print("d) Most expensive:    ", books["price"].max())

In [ ]:
# ── Q3 Answer ────────────────────────────────────────────────────────────────
print("a) Second row (iloc[1]):\n", books.iloc[1])
print("\nb) Last row (iloc[-1]):\n", books.iloc[-1])
print("\nc) Rows 2 to 4 (loc[2:4]):\n", books.loc[2:4])
print("\nd) Title of row 0:", books.loc[0, "title"])

In [ ]:
# ── Q4 Answer ────────────────────────────────────────────────────────────────
print("a) In stock books:\n",      books[books["in_stock"] == True][["title","in_stock"]])
print("\nb) Price below 700:\n",   books[books["price"] < 700][["title","price"]])
print("\nc) Published after 2015:\n", books[books["year"] > 2015][["title","year"]])

In [ ]:
# ── Q5 Answer ────────────────────────────────────────────────────────────────
books_q5 = books.copy()  # work on a copy so we don't affect original

books_q5["discounted_price"] = (books_q5["price"] * 0.85).round(0).astype(int)
books_q5["is_recent"] = books_q5["year"] > 2015
books_q5 = books_q5.drop("in_stock", axis=1)

print(books_q5)

In [ ]:
# ── Q6 Answer ────────────────────────────────────────────────────────────────
sales_raw = pd.DataFrame({
    "Prod Name":  ["Laptop", "Mouse", "Monitor", "Keyboard", "Webcam"],
    "Unit Price": [75000, 1200, 22000, 2500, 4500],
    "Qty Sold":   [5, 120, 8, 45, 30],
})

sales = sales_raw.rename(columns={
    "Prod Name":  "product_name",
    "Unit Price": "unit_price",
    "Qty Sold":   "quantity_sold",
})
sales["total_revenue"] = sales["unit_price"] * sales["quantity_sold"]

high_revenue = sales[sales["total_revenue"] > 10000].sort_values("total_revenue", ascending=False)
print(high_revenue)

In [ ]:
# ── Q7 Answer ────────────────────────────────────────────────────────────────
team = pd.DataFrame({
    "name":       ["Alice","Bob","Carol","Dave","Eve","Frank","Grace"],
    "age":        [28, 34, 26, 41, 30, 35, 27],
    "department": ["Engineering","Marketing","Engineering","HR","Marketing","Engineering","HR"],
    "salary":     [85000, 72000, 91000, 68000, 75000, 95000, 64000],
    "city":       ["Delhi","Mumbai","Pune","Delhi","Bangalore","Mumbai","Pune"],
})

print("a) Engineering + salary > 80K:")
print(team[(team["department"] == "Engineering") & (team["salary"] > 80000)][["name","salary"]])

print("\nb) Delhi or Mumbai:")
print(team[team["city"].isin(["Delhi", "Mumbai"])][["name","city"]])

print("\nc) NOT Marketing:")
print(team[~(team["department"] == "Marketing")][["name","department"]])

print("\nd) Age 25–35:")
print(team[(team["age"] >= 25) & (team["age"] <= 35)][["name","age"]])

In [ ]:
# ── Q8 Answer ────────────────────────────────────────────────────────────────
students_raw = pd.DataFrame({
    "nm":  ["Riya", "Arjun", "Priya", "Karan", "Meera", "Dev", "Sana"],
    "mks": [91, 74, 88, 62, 85, 78, 95],
    "cls": ["10A", "10B", "10A", "10C", "10B", "10A", "10C"],
})

# a) Rename
students = students_raw.rename(columns={"nm": "name", "mks": "marks", "cls": "class_section"})

# b) Add grade
def assign_grade(marks):
    if marks >= 85:
        return "A"
    elif marks >= 70:
        return "B"
    else:
        return "C"

students["grade"] = students["marks"].apply(assign_grade)

# c) Filter grade A
grade_a = students[students["grade"] == "A"]

# d) Drop marks column
final = grade_a.drop("marks", axis=1)

# e) Print
print(final)

---

## Quick Revision

| Topic | Key point |
|---|---|
| DataFrame | 2D table. Each column is a Series with a shared index. |
| Create | From dict of lists, list of dicts, list of lists |
| Inspect | `.head()`, `.tail()`, `.shape`, `.dtypes`, `.info()`, `.describe()` |
| Select column | `df["col"]` → Series. `df[["col1","col2"]]` → DataFrame |
| Select row | `.loc[label]` (inclusive slice) or `.iloc[position]` (exclusive slice) |
| Filter | `df[condition]` using boolean mask. Use `&`, `|`, `~`, not `and`/`or`/`not` |
| Add column | `df["new"] = values` (mutates) or `.assign()` (returns new) |
| Remove column | `.drop("col", axis=1)` — returns new DataFrame |
| Rename | `.rename(columns={"old": "new"})` — returns new DataFrame |

## 3 Interview Questions

**Q1:** What is the difference between `df["salary"]` and `df[["salary"]]`?

**Q2:** When you do `df.drop("city", axis=1)`, does the original DataFrame change? How would you make it permanent?

**Q3:** You have a DataFrame where `df.loc[2]` and `df.iloc[2]` return different rows. How is this possible?

---

## What's Next?

**Notebook 3 — Data Cleaning:** Handling missing values in DataFrames, fixing dtypes, removing duplicates, and replacing values. This is the most important skill in real-world data work.